In [ ]:
import os
from pathlib import Path

import xarray as xr

BASEDIR = Path(os.environ["PROJECT_ROOT"])
input_dir = BASEDIR / "data" / "input"
output_dir = BASEDIR / "data" / "output"

In [3]:

PARAMETERS = ["temperature", "salinity", "u_eastward", "v_northward"]

### Data fetching
Process downloaded NorKyst data (ROMS model for the Norwegian coast).
This data will be used for the boundary conditions, which are set up as forcing.

In [4]:
DATA_DIR = input_dir / "NorKyst2"
# List all .nc files in the folder (non-recursive)
#files = [f.name for f in data_dir.glob("*.nc")]
#print(files)
def list_local_files():
    return [f.name for f in Path(DATA_DIR).glob("*.nc")]


In [5]:
# names are like 20240101, so put an appropriate string to get specific files
month_string = "2020" #"202001"
#files = sorted([s for s in list_opendap_files() if month_string in s])
#urls = [os.path.join(OPENDAP_URL, x) for x in files]
#dss = [xr.open_dataset(url)[PARAMETERS] for url in urls]
files = sorted([name for name in list_local_files() if month_string in name])
paths = [DATA_DIR / name for name in files]
dss = [xr.open_dataset(path)[PARAMETERS] for path in paths]

In [6]:
ds = xr.combine_by_coords(dss, combine_attrs="override")  # downloading, takes a while

In [ ]:
encoding = {var: {"zlib": True, "complevel": 5} for var in ds.data_vars}
ds.to_netcdf(
    os.path.join(input_dir, f"NorKyst-800m_ZDEPTHS_avg_{month_string}.nc"),
    encoding=encoding,
)

In [8]:
del ds, dss, files, encoding, month_string
##del ds, dss, urls, files, encoding, month_string